In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="evaluation",
    notebook_name="03_bleu_rouge.ipynb"
)

# BLEU & ROUGE: Scoring Translations and Summaries

## What You'll Learn

In this notebook, we'll learn two of the most popular metrics for evaluating
text that AI generates:

- **BLEU** -- "How good is this translation?" (used for machine translation)
- **ROUGE** -- "How good is this summary?" (used for text summarization)

We'll:
1. Calculate both metrics **from scratch** so you understand how they work
2. Then use standard libraries (the easy way)
3. Explore their strengths and weaknesses with experiments

No ML knowledge needed!

---
## 1. Setup

In [ ]:
# Install if needed (uncomment the lines below)
# !pip install nltk rouge-score

import math
from collections import Counter
import matplotlib.pyplot as plt

print("Basic setup done!")

# We'll import nltk and rouge-score later when we need them

---
## Part 1: BLEU (Bilingual Evaluation Understudy)

### The Core Idea

BLEU checks: **"How many words and phrases in the AI's output match the reference?"**

Think of it like grading a translation test by checking how many words the
student got right compared to the answer key.

---
### 2. N-grams: The Building Blocks

Before we compute BLEU, we need to understand **n-grams**.

An n-gram is just a group of N consecutive words:
- 1-gram (unigram) = single word
- 2-gram (bigram) = pair of words
- 3-gram (trigram) = triple of words

Let's build a function to extract them:

In [ ]:
def get_ngrams(words, n):
    """
    Extract n-grams from a list of words.

    Example: get_ngrams(["the", "cat", "sat"], 2)
             returns: [("the", "cat"), ("cat", "sat")]
    """
    return [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]


# Let's see it in action
example = "the cat sat on the mat".split()

print(f"Sentence: {' '.join(example)}")
print()
for n in range(1, 5):
    ngrams = get_ngrams(example, n)
    print(f"{n}-grams: {ngrams}")

---
### 3. BLEU From Scratch

Let's build BLEU step by step. The algorithm:

1. For each n (1 to 4), count how many n-grams in the **candidate** (AI output)
   also appear in the **reference** (correct answer)
2. Compute the **precision** for each n (matches / total candidate n-grams)
3. Take the **geometric mean** of all precisions
4. Apply a **brevity penalty** if the candidate is too short

In [ ]:
def compute_ngram_precision(candidate_words, reference_words, n):
    """
    Compute the n-gram precision between a candidate and reference.

    Precision = (matching n-grams) / (total candidate n-grams)

    Uses "clipped" counts: each reference n-gram can only be matched once.
    This prevents gaming by repeating words.
    """
    candidate_ngrams = get_ngrams(candidate_words, n)
    reference_ngrams = get_ngrams(reference_words, n)

    # Count n-grams
    candidate_counts = Counter(candidate_ngrams)
    reference_counts = Counter(reference_ngrams)

    # Clip: each n-gram can match at most as many times as it appears in reference
    clipped_counts = {
        ngram: min(count, reference_counts[ngram])
        for ngram, count in candidate_counts.items()
    }

    numerator = sum(clipped_counts.values())
    denominator = len(candidate_ngrams)

    if denominator == 0:
        return 0.0
    return numerator / denominator


# Example
reference = "the cat is on the mat".split()
candidate = "the cat sat on the mat".split()

print(f"Reference: {' '.join(reference)}")
print(f"Candidate: {' '.join(candidate)}")
print()

for n in range(1, 5):
    precision = compute_ngram_precision(candidate, reference, n)
    ref_ngrams = get_ngrams(reference, n)
    cand_ngrams = get_ngrams(candidate, n)
    print(f"{n}-gram precision: {precision:.2%}")
    print(f"  Candidate {n}-grams: {cand_ngrams}")
    print(f"  Reference {n}-grams: {ref_ngrams}")
    print()

In [ ]:
def brevity_penalty(candidate_words, reference_words):
    """
    Compute the brevity penalty.

    If the candidate is shorter than the reference, we penalize it.
    This prevents cheating by outputting very short translations.

    BP = 1                           if candidate >= reference length
    BP = exp(1 - ref_len/cand_len)   if candidate < reference length
    """
    c = len(candidate_words)
    r = len(reference_words)

    if c >= r:
        return 1.0
    else:
        return math.exp(1 - r / c)


# Examples
print("Brevity Penalty Examples:")
print(f"  Same length (6 vs 6):    BP = {brevity_penalty('a b c d e f'.split(), 'a b c d e f'.split()):.4f}")
print(f"  Slightly short (5 vs 6): BP = {brevity_penalty('a b c d e'.split(), 'a b c d e f'.split()):.4f}")
print(f"  Very short (2 vs 6):     BP = {brevity_penalty('a b'.split(), 'a b c d e f'.split()):.4f}")
print(f"  Just one word (1 vs 6):  BP = {brevity_penalty('a'.split(), 'a b c d e f'.split()):.4f}")

In [ ]:
def compute_bleu(candidate_text, reference_text, max_n=4):
    """
    Compute the BLEU score from scratch.

    Steps:
    1. Compute n-gram precision for n=1,2,3,4
    2. Take geometric mean of log precisions
    3. Multiply by brevity penalty

    Returns: BLEU score between 0 and 1 (higher = better)
    """
    candidate_words = candidate_text.lower().split()
    reference_words = reference_text.lower().split()

    # Step 1: Compute precisions for each n
    precisions = []
    for n in range(1, max_n + 1):
        p = compute_ngram_precision(candidate_words, reference_words, n)
        precisions.append(p)

    # Handle zero precisions (can't take log of 0)
    if any(p == 0 for p in precisions):
        return 0.0

    # Step 2: Geometric mean of precisions (via log)
    log_precision_avg = sum(math.log(p) for p in precisions) / max_n

    # Step 3: Apply brevity penalty
    bp = brevity_penalty(candidate_words, reference_words)

    bleu = bp * math.exp(log_precision_avg)
    return bleu


# Test it!
ref = "the cat is on the mat"
cand = "the cat is on the mat"

print(f"Reference: '{ref}'")
print(f"Candidate: '{cand}'")
print(f"BLEU: {compute_bleu(cand, ref):.4f}")
print()
print("(1.0 = perfect match!)")

---
### 4. BLEU in Action: Comparing Translations

In [ ]:
# Reference (correct) translation
reference = "The cat is sitting on the mat"

# Different candidate translations (from bad to good)
candidates = [
    ("The cat is sitting on the mat", "Perfect match"),
    ("The cat is on the mat", "Close but missing 'sitting'"),
    ("A cat sits on the mat", "Same meaning, different words"),
    ("The dog is sitting on the mat", "Wrong animal!"),
    ("Mat the on sitting is cat the", "Right words, wrong order"),
    ("I like pizza", "Completely wrong"),
]

print(f"Reference: '{reference}'")
print("=" * 70)

names = []
scores = []

for candidate, description in candidates:
    bleu = compute_bleu(candidate, reference)
    names.append(description)
    scores.append(bleu)
    bar = "█" * int(bleu * 50)
    print(f"\n  BLEU: {bleu:.4f} {bar}")
    print(f"  '{candidate}'")
    print(f"  ({description})")

In [ ]:
# Visualize the comparison
colors = ["#4CAF50" if s > 0.4 else "#FFC107" if s > 0.1 else "#F44336" for s in scores]

plt.figure(figsize=(12, 5))
bars = plt.barh(names, scores, color=colors, edgecolor="black", linewidth=0.5)
plt.xlabel("BLEU Score (0 to 1, higher = better)", fontsize=12)
plt.title("BLEU Scores for Different Translations", fontsize=14, fontweight="bold")
plt.xlim(0, 1.1)

for bar, score in zip(bars, scores):
    plt.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
             f"{score:.3f}", va="center", fontweight="bold")

plt.tight_layout()
plt.show()

---
### 5. Using NLTK's BLEU (The Easy Way)

In practice, you don't compute BLEU from scratch. The `nltk` library has a
well-tested implementation:

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# NLTK expects tokenized lists
reference_tokens = "The cat is sitting on the mat".lower().split()
candidate_tokens = "The cat is on the mat".lower().split()

# sentence_bleu expects reference as a list of lists (supports multiple references)
bleu_nltk = sentence_bleu(
    [reference_tokens],   # list of reference(s)
    candidate_tokens,     # candidate
    smoothing_function=SmoothingFunction().method1  # handles zero n-gram counts
)

print(f"NLTK BLEU Score: {bleu_nltk:.4f}")
print()
print("That's it! One function call. Much easier than doing it from scratch.")

---
## Part 2: ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

### The Core Idea

While BLEU focuses on **precision** ("How much of the AI output is correct?"),
ROUGE focuses on **recall** ("How much of the reference did the AI capture?").

This makes ROUGE ideal for **summarization** -- where we want to make sure
the AI included all the important points.

```
BLEU:  "Of the words the AI used, how many match the reference?"
ROUGE: "Of the words in the reference, how many appear in the AI output?"
```

---
### 6. ROUGE From Scratch

There are three main variants:
- **ROUGE-1**: Compare single words (unigrams)
- **ROUGE-2**: Compare word pairs (bigrams)
- **ROUGE-L**: Find the Longest Common Subsequence

In [ ]:
def compute_rouge_n(candidate_text, reference_text, n=1):
    """
    Compute ROUGE-N (N=1 for unigrams, N=2 for bigrams).

    Returns: dict with precision, recall, and F1 scores
    """
    candidate_words = candidate_text.lower().split()
    reference_words = reference_text.lower().split()

    candidate_ngrams = Counter(get_ngrams(candidate_words, n))
    reference_ngrams = Counter(get_ngrams(reference_words, n))

    # Count overlapping n-grams
    overlap = sum(
        min(candidate_ngrams[ng], reference_ngrams[ng])
        for ng in candidate_ngrams
        if ng in reference_ngrams
    )

    # Precision: overlap / candidate n-gram count
    precision = overlap / sum(candidate_ngrams.values()) if candidate_ngrams else 0

    # Recall: overlap / reference n-gram count
    recall = overlap / sum(reference_ngrams.values()) if reference_ngrams else 0

    # F1: harmonic mean of precision and recall
    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


# Example
reference = "The president signed the new climate bill into law today"
candidate = "The president signed a new bill on climate change"

print(f"Reference: '{reference}'")
print(f"Candidate: '{candidate}'")
print()

for n in [1, 2]:
    result = compute_rouge_n(candidate, reference, n)
    print(f"ROUGE-{n}:")
    print(f"  Precision: {result['precision']:.2%} (of candidate words, how many match?)")
    print(f"  Recall:    {result['recall']:.2%} (of reference words, how many were captured?)")
    print(f"  F1:        {result['f1']:.2%} (balanced score)")
    print()

In [ ]:
def longest_common_subsequence(seq1, seq2):
    """
    Find the Longest Common Subsequence (LCS) between two sequences.

    LCS finds the longest sequence of elements that appear in the same
    ORDER in both sequences (but not necessarily consecutively).

    Example:
      seq1 = ["the", "cat", "was", "sitting", "on", "a", "mat"]
      seq2 = ["the", "cat", "sat", "on", "the", "mat"]
      LCS  = ["the", "cat", "on", "mat"] (length 4)
    """
    m, n = len(seq1), len(seq2)
    # Build a table of LCS lengths
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if seq1[i-1] == seq2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    return dp[m][n]


def compute_rouge_l(candidate_text, reference_text):
    """
    Compute ROUGE-L using the Longest Common Subsequence.
    """
    candidate_words = candidate_text.lower().split()
    reference_words = reference_text.lower().split()

    lcs_length = longest_common_subsequence(candidate_words, reference_words)

    precision = lcs_length / len(candidate_words) if candidate_words else 0
    recall = lcs_length / len(reference_words) if reference_words else 0

    if precision + recall == 0:
        f1 = 0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1, "lcs_length": lcs_length}


# Example
result_l = compute_rouge_l(candidate, reference)
print(f"ROUGE-L:")
print(f"  LCS length: {result_l['lcs_length']} words in common order")
print(f"  Precision:  {result_l['precision']:.2%}")
print(f"  Recall:     {result_l['recall']:.2%}")
print(f"  F1:         {result_l['f1']:.2%}")

---
### 7. ROUGE in Action: Comparing Summaries

In [ ]:
# Reference summary (the "gold standard" written by a human)
reference_summary = (
    "The researchers discovered a new species of deep sea fish "
    "that can produce its own light using bioluminescence"
)

# Different AI-generated summaries
summaries = [
    ("The researchers discovered a new species of deep sea fish that can produce its own light using bioluminescence",
     "Perfect match"),
    ("Scientists found a new deep sea fish species that produces light through bioluminescence",
     "Good -- same meaning, different words"),
    ("A new fish was found in the deep sea",
     "Partial -- misses key details"),
    ("Researchers study ocean creatures",
     "Vague -- too general"),
    ("The weather will be sunny tomorrow",
     "Wrong -- completely off topic"),
]

print(f"Reference: '{reference_summary}'")
print("=" * 80)

labels = []
rouge1_scores = []
rouge2_scores = []
rougel_scores = []

for summary, description in summaries:
    r1 = compute_rouge_n(summary, reference_summary, 1)
    r2 = compute_rouge_n(summary, reference_summary, 2)
    rl = compute_rouge_l(summary, reference_summary)

    labels.append(description)
    rouge1_scores.append(r1["f1"])
    rouge2_scores.append(r2["f1"])
    rougel_scores.append(rl["f1"])

    print(f"\n  '{summary}'")
    print(f"  ({description})")
    print(f"  ROUGE-1 F1: {r1['f1']:.2%} | ROUGE-2 F1: {r2['f1']:.2%} | ROUGE-L F1: {rl['f1']:.2%}")

In [ ]:
# Visualize ROUGE scores
import numpy as np

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
bars1 = ax.bar(x - width, rouge1_scores, width, label="ROUGE-1", color="#2196F3")
bars2 = ax.bar(x, rouge2_scores, width, label="ROUGE-2", color="#FF9800")
bars3 = ax.bar(x + width, rougel_scores, width, label="ROUGE-L", color="#4CAF50")

ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("ROUGE Scores for Different Summaries", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.legend()
ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("Key observations:")
print("- ROUGE-1 is most forgiving (just checks individual words)")
print("- ROUGE-2 is stricter (requires matching word PAIRS)")
print("- ROUGE-L checks word order through longest common subsequence")

---
### 8. Using the `rouge-score` Library (The Easy Way)

In [ ]:
from rouge_score import rouge_scorer

# Create a scorer for ROUGE-1, ROUGE-2, and ROUGE-L
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

# Score a summary
reference = "The researchers discovered a new species of deep sea fish that produces light"
candidate = "Scientists found a new deep sea fish species that makes its own light"

scores = scorer.score(reference, candidate)

print(f"Reference: '{reference}'")
print(f"Candidate: '{candidate}'")
print()
for metric, score in scores.items():
    print(f"{metric}:")
    print(f"  Precision: {score.precision:.2%}")
    print(f"  Recall:    {score.recall:.2%}")
    print(f"  F1:        {score.fmeasure:.2%}")
    print()

print("Note: 'use_stemmer=True' means the library treats 'running' and 'run'")
print("as the same word. This gives more forgiving scores.")

---
## Part 3: Limitations and Gotchas

### 9. Where BLEU and ROUGE Fail

Both metrics compare **exact words**. They don't understand **meaning**.
This leads to some surprising failures:

In [ ]:
# FAILURE 1: Synonyms get penalized!
ref = "The dog is happy"

# Same meaning, different words
synonym = "The puppy is joyful"

# Different meaning, similar words
misleading = "The dog is not happy"

rouge1_synonym = compute_rouge_n(synonym, ref, 1)
rouge1_misleading = compute_rouge_n(misleading, ref, 1)

print("FAILURE: Synonyms vs. Negation")
print("=" * 50)
print(f"Reference:    '{ref}'")
print(f"\nSynonym:      '{synonym}'")
print(f"  ROUGE-1 F1: {rouge1_synonym['f1']:.2%}")
print(f"  (Same meaning but low score because words differ!)")
print(f"\nNegation:     '{misleading}'")
print(f"  ROUGE-1 F1: {rouge1_misleading['f1']:.2%}")
print(f"  (OPPOSITE meaning but higher score because more words match!)")
print()
print("This is a major limitation: word matching != understanding meaning!")

In [ ]:
# FAILURE 2: Word order can be ignored by ROUGE-1
ref = "The cat ate the mouse"
reversed_meaning = "The mouse ate the cat"  # Very different meaning!

r1 = compute_rouge_n(reversed_meaning, ref, 1)
r2 = compute_rouge_n(reversed_meaning, ref, 2)

print("FAILURE: Word order matters for meaning!")
print("=" * 50)
print(f"Reference: '{ref}'")
print(f"Reversed:  '{reversed_meaning}'")
print(f"\nROUGE-1 F1: {r1['f1']:.2%} (looks perfect -- same words!)")
print(f"ROUGE-2 F1: {r2['f1']:.2%} (catches the problem -- word PAIRS differ!)")
print()
print("Lesson: ROUGE-2 and ROUGE-L are better at catching word order issues")
print("than ROUGE-1, but none of them truly understand meaning.")

---
## 10. BLEU vs ROUGE: Side-by-Side Comparison

In [ ]:
# Let's compute both BLEU and ROUGE for the same pairs
ref = "The quick brown fox jumps over the lazy dog"

candidates = [
    "The quick brown fox jumps over the lazy dog",    # Perfect
    "The quick brown fox leaps over the lazy dog",    # One word different
    "A fast brown fox jumps over a sleepy dog",       # Several words different
    "The fox jumps over the dog",                      # Missing words (shorter)
    "The quick brown fox jumps over the lazy dog and then runs away",  # Extra words (longer)
]

print(f"Reference: '{ref}'")
print("=" * 80)

bleu_scores = []
rouge_f1_scores = []
short_labels = ["Perfect", "1 word diff", "Several diff", "Shorter", "Longer"]

for cand, label in zip(candidates, short_labels):
    bleu = compute_bleu(cand, ref)
    rouge = compute_rouge_n(cand, ref, 1)
    bleu_scores.append(bleu)
    rouge_f1_scores.append(rouge["f1"])
    print(f"\n  [{label}] '{cand}'")
    print(f"  BLEU: {bleu:.4f}  |  ROUGE-1 Precision: {rouge['precision']:.2%}  Recall: {rouge['recall']:.2%}  F1: {rouge['f1']:.2%}")

# Visualize
x = np.arange(len(short_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width/2, bleu_scores, width, label="BLEU", color="#2196F3")
ax.bar(x + width/2, rouge_f1_scores, width, label="ROUGE-1 F1", color="#FF5722")
ax.set_ylabel("Score")
ax.set_title("BLEU vs ROUGE-1 on the Same Text Pairs", fontsize=14, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(short_labels)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

print("\nKey difference:")
print("- BLEU penalizes shorter outputs (brevity penalty)")
print("- ROUGE's recall is more forgiving of shorter outputs if they hit key words")
print("- BLEU checks precision; ROUGE checks both precision and recall")

---
## 11. Experiment: Try Your Own!

Change the reference and candidate texts below and see how the scores change.

---
## What You Just Built

You implemented BLEU and ROUGE from scratch and verified them against standard libraries:
- **BLEU:** n-gram precision with clipping, geometric mean, brevity penalty
- **ROUGE-N:** n-gram recall (and precision and F1) for unigrams and bigrams
- **ROUGE-L:** Longest Common Subsequence via dynamic programming
- **Limitations:** synonym blindness, word-order insensitivity in ROUGE-1

For the full formulas (modified precision derivation, brevity penalty math, LCS algorithm), failure mode analysis, BERTScore/METEOR/CIDEr alternatives, and staff-level interview questions, see the [interview deep-dive](./bleu-rouge-interview.md).

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)

---
## 12. Summary

```
┌───────────────────────────────────────────────────────────┐
│                BLEU & ROUGE Cheat Sheet                   │
│                                                           │
│  BLEU:                                                    │
│    Focus: Precision (is the output correct?)              │
│    Best for: Machine translation                          │
│    Uses: N-gram matching + brevity penalty                │
│                                                           │
│  ROUGE:                                                   │
│    Focus: Recall (did it capture the key points?)         │
│    Best for: Text summarization                           │
│    Variants: ROUGE-1 (words), ROUGE-2 (pairs),           │
│              ROUGE-L (longest common subsequence)         │
│                                                           │
│  Limitations:                                             │
│    - Word matching != meaning understanding               │
│    - Synonyms get penalized unfairly                      │
│    - Use alongside human evaluation                       │
└───────────────────────────────────────────────────────────┘
```

### Next Steps

- Read the [BLEU & ROUGE guide](../metrics/bleu-rouge.md) for more context
- Try the [Benchmarks notebook](./04_benchmarks_demo.ipynb) to see evaluation at scale